# 🚢 Logistics Shipment ETA & Delay Risk Prediction Pipeline
### Aiqon Take-Home Assignment

This interactive notebook executes the end-to-end Machine Learning training pipeline:
1. **Data Ingestion & Cleaning**: Loads raw shipment records and deterministic pre-departure features.
2. **Feature Engineering & Train/Test Split**: Builds feature matrix and splits training/evaluation sets.
3. **ETA Regression Training**: Benchmarks Baseline Mean, Ridge Regression, and Random Forest Regressor.
4. **Delay Risk Classification**: Trains Logistic Regression and Random Forest Classifiers with CV.
5. **Model Persistence**: Serializes best models and metadata into `models/`.
6. **Pipeline Verification**: Validates `PredictionService` loading for the web application (`app.py`).

In [1]:
# --- 1. Environment & DLL Protection Setup ---
import os
import sys
import types
import json
import numpy as np
import pandas as pd

# On Windows, prevent Application Control policies from blocking optional sklearn native modules
if sys.platform == 'win32':
    for _mod in ['sklearn.svm._libsvm_sparse', 'sklearn.svm._libsvm', 'sklearn.svm._liblinear']:
        if _mod not in sys.modules:
            try:
                __import__(_mod)
            except (ImportError, OSError):
                sys.modules[_mod] = types.ModuleType(_mod)

# Determine project root robustly (supports VS Code, Jupyter, terminal, and Google Colab)
curr_dir = os.getcwd()
if os.path.exists(os.path.join(curr_dir, 'src')):
    project_root = curr_dir
elif os.path.exists(os.path.join(curr_dir, '..', 'src')):
    project_root = os.path.abspath(os.path.join(curr_dir, '..'))
else:
    project_root = curr_dir

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'✓ Project root configured: {project_root}')
print(f'✓ Working directory: {os.getcwd()}')

# Core imports from project src
from sklearn.model_selection import train_test_split
from src import CONFIG, logger
from src.data_loader import load_raw_data
from src.preprocessing import preprocess, save_processed
from src.feature_engineering import (
    build_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
)
from src.eta_model import train_eta_models, save_model
from src.delay_classifier import train_classifiers, save_classifier
print('✓ All project modules and scikit-learn components imported successfully!')

✓ Project root configured: C:\Users\joker\Desktop\test2
✓ Working directory: C:\Users\joker\Desktop\test2
✓ All project modules and scikit-learn components imported successfully!


## Phase 1: Load and Preprocess Raw Shipment Data
Loads raw Excel file, parses UTC timestamps, checks for target discrepancies, and exports cleaned data.

In [2]:
seed = CONFIG['project']['random_seed']
test_size = CONFIG['data']['test_size']

print('--- Loading Raw Data ---')
raw_df = load_raw_data()
print(f'Raw dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')

print('--- Running Preprocessing Pipeline ---')
cleaned_df = preprocess(raw_df)
save_processed(cleaned_df)

# Filter to records with observed outcome for supervised learning
df_train_pool = cleaned_df[cleaned_df['has_outcome'] == True].copy()
print(f'Records with observed outcome for model training: {len(df_train_pool)}')
print(f'Records in-transit (missing final arrival, outcome false): {len(cleaned_df) - len(df_train_pool)}')
df_train_pool[['shipment_id', 'origin_city', 'destination_city', 'delay_target_minutes', 'is_delayed']].head()

2026-10-12 12:01:31,918 [INFO] shipment_eta: Loading raw data from C:\Users\joker\Desktop\test2\data/raw/logistics-shipment-delay-prediction.xlsx


--- Loading Raw Data ---


2026-10-12 12:01:32,370 [INFO] shipment_eta: Loaded 200 rows, 25 columns


2026-10-12 12:01:32,390 [WARNING] shipment_eta: 13 records have >1 min discrepancy between delay_actual_minutes and timestamp-derived delay


2026-10-12 12:01:32,413 [INFO] shipment_eta: Preprocessing complete: 200 rows, 145 with outcome, 55 without


2026-10-12 12:01:32,427 [INFO] shipment_eta: Processed data saved to C:\Users\joker\Desktop\test2\data/processed/shipments_cleaned.csv


Raw dataset shape: 200 rows, 25 columns
--- Running Preprocessing Pipeline ---
Records with observed outcome for model training: 145
Records in-transit (missing final arrival, outcome false): 55


,shipment_id,origin_city,destination_city,delay_target_minutes,is_delayed
0,SHIP0001,chicago,st. louis,17.0,1.0
1,SHIP0002,paris,berlin,24.0,1.0
3,SHIP0004,mumbai,dubai,77.0,1.0
4,SHIP0005,dallas,houston,27.0,1.0
5,SHIP0006,tokyo,osaka,9.0,1.0


## Phase 2: Feature Matrix Construction & Train/Test Split
Constructs pre-departure feature matrix (preventing leakage from arrival/delay columns) and creates a stratified split.

In [3]:
X_all, num_cols, cat_cols = build_feature_matrix(df_train_pool)
y_reg = df_train_pool['delay_target_minutes'].values
y_clf = df_train_pool['is_delayed'].astype(int).values

print(f'Numeric features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_all, y_reg, y_clf,
    test_size=test_size,
    random_state=seed,
    stratify=y_clf,
)

print(f'✓ Train set size: {len(X_train)} samples')
print(f'✓ Test set size:  {len(X_test)} samples')

2026-10-12 12:01:32,465 [INFO] shipment_eta: Feature matrix: 7 numeric + 5 categorical = 12 features


Numeric features (7): ['route_distance_km', 'carrier_on_time_rate', 'scheduled_duration_min', 'departure_delay_min', 'hour_of_departure', 'day_of_week', 'is_international']
Categorical features (5): ['route_type', 'vehicle_type', 'weather_condition_departure', 'priority_level', 'shipment_type']
✓ Train set size: 116 samples
✓ Test set size:  29 samples


## Phase 3: Train ETA Regression Models
Trains and cross-validates Baseline Mean, Ridge Regression, and Random Forest Regressor.

In [4]:
print('=== Training ETA Regression Candidates ===')
eta_result = train_eta_models(
    X_train, y_reg_train,
    preprocessor=None,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_eta_name = eta_result['best_name']
best_eta_model = eta_result['models'][best_eta_name]
feature_cols = num_cols + cat_cols

# Save best regression model
save_model(
    best_eta_model,
    best_eta_name,
    eta_result['cv_results'][best_eta_name],
    feature_cols,
)

print('\n--- ETA Model Benchmark Summary ---')
eta_summary = pd.DataFrame(eta_result['cv_results']).T
display(eta_summary)
print(f'★ Best ETA Regressor: {best_eta_name}')

=== Training ETA Regression Candidates ===


2026-10-12 12:01:32,700 [INFO] shipment_eta: Baseline MAE: 34.31 ± 5.49


2026-10-12 12:01:33,634 [INFO] shipment_eta: Ridge MAE: 20.09 ± 4.41 (alpha=1.0)


2026-10-12 12:01:40,846 [INFO] shipment_eta: RF MAE: 21.35 ± 3.33


2026-10-12 12:01:40,847 [INFO] shipment_eta: Best ETA model by CV MAE: ridge (20.09)


2026-10-12 12:01:40,852 [INFO] shipment_eta: ETA model saved to C:\Users\joker\Desktop\test2\models\eta_model.joblib



--- ETA Model Benchmark Summary ---


,mae_mean,mae_std,best_alpha,best_params
baseline_mean,34.312588,5.492591,NaN,NaN
ridge,20.091265,4.406147,1.0,NaN
random_forest,21.354893,3.328123,NaN,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."


★ Best ETA Regressor: ridge


## Phase 4: Train Delay Risk Classification Models
Trains and cross-validates Logistic Regression and Random Forest Classifiers.

In [5]:
print('=== Training Delay Risk Classifiers ===')
clf_result = train_classifiers(
    X_train, y_clf_train,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_clf_name = clf_result['best_name']
best_clf_model = clf_result['models'][best_clf_name]

# Save best classification model
save_classifier(
    best_clf_model,
    best_clf_name,
    clf_result['cv_results'][best_clf_name],
    feature_cols,
)

print('\n--- Delay Classifier Benchmark Summary ---')
clf_summary = pd.DataFrame(clf_result['cv_results']).T
display(clf_summary)
print(f'★ Best Delay Classifier: {best_clf_name}')

C:\Users\joker\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\joker\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


=== Training Delay Risk Classifiers ===


C:\Users\joker\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


2026-10-12 12:01:41,174 [INFO] shipment_eta: LR F1: 0.914 ± 0.035  |  ROC-AUC: nan


C:\Users\joker\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


...

2026-10-12 12:01:43,091 [INFO] shipment_eta: Best classifier: random_forest (F1=0.949)


2026-10-12 12:01:43,122 [INFO] shipment_eta: Classifier saved to C:\Users\joker\Desktop\test2\models\delay_classifier.joblib



--- Delay Classifier Benchmark Summary ---


,f1_mean,f1_std,roc_auc_mean,roc_auc_std,best_params
logistic_regression,0.913952,0.034501,NaN,NaN,NaN
random_forest,0.948812,0.028063,NaN,NaN,"{'max_depth': 5, 'min_samples_leaf': 3, 'n_est..."


★ Best Delay Classifier: random_forest


## Phase 5: Persist Train/Test Split Metadata
Saves split indices so offline evaluation scripts can evaluate test generalization without leakage.

In [6]:
split_info = {
    'train_indices': X_train.index.tolist(),
    'test_indices': X_test.index.tolist(),
}
split_path = os.path.join(project_root, CONFIG['paths']['models_dir'], 'split_info.json')
os.makedirs(os.path.dirname(split_path), exist_ok=True)
with open(split_path, 'w', encoding='utf-8') as f:
    json.dump(split_info, f, indent=2)
print(f'✓ Train/Test split metadata persisted to: {split_path}')

✓ Train/Test split metadata persisted to: C:\Users\joker\Desktop\test2\models\split_info.json


## Phase 6: End-to-End Prediction Service & `app.py` Verification
Tests that `PredictionService` instantiates cleanly and loads both serialized models for use in `app.py`.

In [7]:
from src.prediction_service import PredictionService

service = PredictionService()
print('--- Service Health Check ---')
print('✓ ETA Regression Pipeline loaded:', service.eta_model is not None)
print('✓ Delay Classifier Pipeline loaded:', service.clf_model is not None)

# Test live prediction on a sample test shipment
sample_record = df_train_pool.iloc[0].to_dict()
pred = service.predict_single(sample_record)

print('\n--- Live Prediction Sample Test ---')
print(f'Shipment ID:                {sample_record.get("shipment_id")}')
print(f'Origin -> Destination:      {sample_record.get("origin_city")} -> {sample_record.get("destination_city")}')
print(f'Predicted Delay (minutes):  {pred["predicted_delay_min"]:.2f}')
print(f'Delay Risk Category:        {pred["delay_risk"]}')
print(f'Delay Probability:          {pred["delay_probability"] * 100:.1f}%')
print(f'Stages Simulated:           {len(pred["propagation"].get("stages", []))}')

print('=' * 65)
print('🎉 SUCCESS: All models trained and verified without errors!')
print('🚀 You can now run `python app.py` to start the Flask Web App!')
print('=' * 65)

2026-10-12 12:01:43,160 [INFO] shipment_eta: ETA model loaded: ridge


2026-10-12 12:01:43,196 [INFO] shipment_eta: Classifier loaded: random_forest


--- Service Health Check ---
✓ ETA Regression Pipeline loaded: True
✓ Delay Classifier Pipeline loaded: True



--- Live Prediction Sample Test ---
Shipment ID:                SHIP0001
Origin -> Destination:      chicago -> st. louis
Predicted Delay (minutes):  13.90
Delay Risk Category:        Delayed
Delay Probability:          56.2%
Stages Simulated:           4
🎉 SUCCESS: All models trained and verified without errors!
🚀 You can now run `python app.py` to start the Flask Web App!
